In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "processed"

sales = pd.read_parquet(DATA / "sales.parquet")
sales["series"] = sales["store_id"] + "/" + sales["item_id"]
sales["is_zero"] = sales["units"] == 0
print(sales.shape)
sales.dtypes

In [ ]:
profile = sales.groupby("series").agg(
    start=("date", "min"),
    days=("date", "count"),
    mean_units=("units", "mean"),
    zero_share=("is_zero", "mean"),
)
profile.describe()

In [ ]:
picks = {
    "fastest": profile["mean_units"].idxmax(),
    "median": profile["mean_units"].sort_values().index[len(profile) // 2],
    "slowest": profile["mean_units"].idxmin(),
}

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, (label, name) in zip(axes, picks.items(), strict=True):
    series = sales[sales["series"] == name].set_index("date")["units"]
    ax.plot(series.index, series.values, linewidth=0.6)
    ax.set_title(f"{label}: {name}")
plt.tight_layout()

In [ ]:
sales["day"] = sales["d"].str[2:].astype(int)

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = sales.groupby(sales["date"].dt.day_name())["units"].mean().reindex(order)
print((by_weekday / sales["units"].mean()).round(2))

In [ ]:
first_day = sales.groupby("series")["day"].min()

for origin in [1829, 1857, 1885, 1913]:
    enough = (first_day <= origin - 56).sum()
    print(f"origin d_{origin}: {enough} of {len(first_day)} series have 8+ weeks of history")

In [ ]:
import numpy as np

wide = sales.pivot(index="series", columns="day", values="units")
wide = wide.reindex(columns=range(1, 1942))
print(wide.shape)
wide.iloc[:3, :5]

In [ ]:
HORIZON = 28
ORIGIN = 1913
MIN_HISTORY = 56

history = wide.loc[:, :ORIGIN]
actual = wide.loc[:, ORIGIN + 1 : ORIGIN + HORIZON]

first_day = history.notna().idxmax(axis=1)
eligible = first_day <= ORIGIN - MIN_HISTORY
history, actual = history[eligible], actual[eligible]

assert not actual.isna().any().any()
print(history.shape, actual.shape)

In [ ]:
def forecast_zero(hist, h):
    return np.zeros((len(hist), h))


def forecast_naive(hist, h):
    last = hist.iloc[:, -1].to_numpy()
    return np.tile(last[:, None], (1, h))


def forecast_seasonal_naive(hist, h):
    last_week = hist.iloc[:, -7:].to_numpy()
    return np.tile(last_week, (1, int(np.ceil(h / 7))))[:, :h]


def forecast_moving_average(hist, h, window=28):
    mean = hist.iloc[:, -window:].mean(axis=1).to_numpy()
    return np.tile(mean[:, None], (1, h))

In [ ]:
def score(forecast, actual):
    error = forecast - actual.to_numpy()
    return {
        "MAE": np.abs(error).mean(),
        "RMSE": np.sqrt((error**2).mean()),
        "bias": error.mean(),
    }


methods = {
    "zero": forecast_zero,
    "naive": forecast_naive,
    "seasonal naive": forecast_seasonal_naive,
    "moving avg 28d": forecast_moving_average,
}

results = {name: score(fn(history, HORIZON), actual) for name, fn in methods.items()}
pd.DataFrame(results).T.round(3)

In [ ]:
FOLDS = [1829, 1857, 1885, 1913]

rows = []
for origin in FOLDS:
    hist = wide.loc[:, :origin]
    act = wide.loc[:, origin + 1 : origin + HORIZON]
    first = hist.notna().idxmax(axis=1)
    keep = first <= origin - MIN_HISTORY
    hist, act = hist[keep], act[keep]
    assert not act.isna().any().any()

    for name, fn in methods.items():
        result = score(fn(hist, HORIZON), act)
        rows.append({"origin": origin, "method": name, "n_series": len(hist), **result})

folds = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "bias"]:
    table = folds.pivot(index="method", columns="origin", values=metric)
    table["mean"] = table.mean(axis=1)
    print(metric)
    print(table.round(3), "\n")

In [ ]:
from ml.metrics import bias, mae, mase, rmse, rmsse

rows = []
for origin in FOLDS:
    hist = wide.loc[:, :origin]
    act = wide.loc[:, origin + 1 : origin + HORIZON]
    keep = hist.notna().idxmax(axis=1) <= origin - MIN_HISTORY
    hist, act = hist[keep], act[keep]
    h, a = hist.to_numpy(dtype=float), act.to_numpy(dtype=float)

    for name, fn in methods.items():
        fc = fn(hist, HORIZON)
        rows.append(
            {
                "origin": origin,
                "method": name,
                "MAE": mae(fc, a),
                "RMSE": rmse(fc, a),
                "MASE": mase(fc, a, h),
                "RMSSE": rmsse(fc, a, h),
            }
        )

scaled = pd.DataFrame(rows)
for metric in ["MASE", "RMSSE"]:
    table = scaled.pivot(index="method", columns="origin", values=metric)
    table["mean"] = table.mean(axis=1)
    print(metric)
    print(table.round(3), "\n")

In [ ]:
from ml.backtest import make_grid, run_backtest, summarize
from ml.baselines import BASELINES

grid = make_grid(sales)
results = run_backtest(grid, BASELINES)
for metric in ["MAE", "RMSE", "MASE", "RMSSE"]:
    print(metric)
    print(summarize(results, metric), "\n")

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, CrostonSBA, TSB, WindowAverage

from ml.backtest import split_fold
from ml.metrics import mae, mase, rmse, rmsse

ORIGIN = 1913
values = grid.to_numpy(dtype=float)
history, actual = split_fold(values, ORIGIN)

observed = ~np.isnan(values[:, :ORIGIN])
first = observed.argmax(axis=1) + 1
names = grid.index[first <= ORIGIN - MIN_HISTORY]
assert len(names) == history.shape[0]

train = sales[(sales["day"] <= ORIGIN) & sales["series"].isin(names)]
train = train[["series", "date", "units"]].rename(
    columns={"series": "unique_id", "date": "ds", "units": "y"}
)
train["y"] = train["y"].astype(float)

sf = StatsForecast(
    models=[
        WindowAverage(window_size=28),
        AutoETS(season_length=7),
        CrostonSBA(),
        TSB(alpha_d=0.2, alpha_p=0.2),
    ],
    freq="D",
)
fc = sf.forecast(df=train, h=HORIZON)
fc.head()

In [ ]:
def to_array(fc, column):
    wide_fc = fc.pivot(index="unique_id", columns="ds", values=column)
    return wide_fc.loc[names].to_numpy().clip(min=0)


rows = []
for model in ["WindowAverage", "AutoETS", "CrostonSBA", "TSB"]:
    forecast = to_array(fc, model)
    rows.append(
        {
            "method": model,
            "MAE": mae(forecast, actual),
            "RMSE": rmse(forecast, actual),
            "MASE": mase(forecast, actual, history),
            "RMSSE": rmsse(forecast, actual, history),
        }
    )

pd.DataFrame(rows).set_index("method").round(3)

In [ ]:
import time

from ml.backtest import make_grid, run_backtest, summarize
from ml.baselines import BASELINES
from ml.stats_baselines import STATS_BASELINES

grid = make_grid(sales)

start = time.time()
stats_results = run_backtest(grid, STATS_BASELINES)
print(f"stats models took {(time.time() - start) / 60:.1f} minutes")

base_results = run_backtest(grid, BASELINES)
all_results = pd.concat([base_results, stats_results], ignore_index=True)

for metric in ["RMSSE", "MASE", "RMSE"]:
    print(metric)
    print(summarize(all_results, metric), "\n")

In [ ]:
cal = pd.read_parquet(DATA / "calendar.parquet").set_index("d")

for origin in (1421, 1785):
    start = cal.loc[f"d_{origin + 1}", "date"].date()
    end = cal.loc[f"d_{origin + 28}", "date"].date()
    print(origin, start, "->", end)

In [ ]:
EXTRA = (1421, 1785)

extra = pd.concat(
    [
        run_backtest(grid, BASELINES, folds=EXTRA),
        run_backtest(grid, STATS_BASELINES, folds=EXTRA),
    ],
    ignore_index=True,
)

print(extra.drop_duplicates("origin")[["origin", "n_series"]])
for metric in ["RMSSE", "RMSE"]:
    print(metric)
    print(summarize(extra, metric), "\n")